# Phase 2: Model Training, Evaluation & Cost-Sensitive Threshold Calibration
### Customer Churn Decision Engine
**Objective:** Compare Logistic Regression vs. Random Forest, analyze class imbalance handling, evaluate Precision-Recall curves, and optimize the decision threshold based on asymmetric business costs.

In [ ]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve, confusion_matrix, ConfusionMatrixDisplay

from src.config import RAW_DATA_DIR
from src.data.preprocessor import load_raw_dataset, split_data
from src.models.train import get_candidate_models
from src.models.pipeline import build_churn_model_pipeline
from src.models.evaluate import compute_metrics_at_threshold, find_optimal_threshold

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Load Data
X, y = load_raw_dataset()
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {len(X_train):,} | Test instances: {len(X_test):,}")

## 1. Model Tournament: Baseline vs. Champion

In [ ]:
candidates = get_candidate_models()
results = {}

for name, clf in candidates.items():
    pipe = build_churn_model_pipeline(classifier=clf)
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    results[name] = {"pipeline": pipe, "probs": probs}
    metrics = compute_metrics_at_threshold(y_test.values, probs, threshold=0.5)
    print(f"=== {name} (Default Threshold = 0.5) ===")
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f} | PR-AUC: {metrics['pr_auc']:.4f}")
    print(f"  Recall:  {metrics['recall']:.4f} | Precision: {metrics['precision']:.4f} | F1: {metrics['f1']:.4f}")

## 2. Precision-Recall & ROC Curve Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, data in results.items():
    fpr, tpr, _ = roc_curve(y_test, data["probs"])
    prec, rec, _ = precision_recall_curve(y_test, data["probs"])
    axes[0].plot(fpr, tpr, label=name, lw=2)
    axes[1].plot(rec, prec, label=name, lw=2)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_title("ROC Curves", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].set_title("Precision-Recall Curves", fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Decision Threshold Optimization & Business Loss Minimization

In [ ]:
rf_probs = results["RandomForest_Champion"]["probs"]
opt_search = find_optimal_threshold(y_test.values, rf_probs, cost_fn=500.0, cost_fp=35.0)
curve_df = pd.DataFrame(opt_search["threshold_curve"])

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(curve_df["threshold"], curve_df["recall"], label="Recall", color="#e74c3c", lw=2)
ax1.plot(curve_df["threshold"], curve_df["precision"], label="Precision", color="#2ecc71", lw=2)
ax1.plot(curve_df["threshold"], curve_df["f2"], label="F2 Score (β=2)", color="#3498db", lw=2.5, linestyle="--")
ax1.axvline(opt_search["best_f2_threshold"], color="#9b59b6", linestyle=":", label=f"Optimal τ* ({opt_search['best_f2_threshold']})")
ax1.set_xlabel("Decision Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Trade-Off Analysis: Precision, Recall, F2 vs. Decision Threshold", fontweight="bold")
ax1.legend(loc="center left")
plt.tight_layout()
plt.show()